# GNN scores for every LI-Small validation and test transaction (AML pipeline V2)

Run top to bottom on a Colab **L4** GPU (Runtime > Change runtime type). Estimate, not measured: about 25-35 min, about 0.5 L4 hours (~1 compute unit). Most of it is building the Multi-GNN environment (10-20 min). Loading the cached graph takes about 1.5 min, and scoring validation and test about 3 min (the Phase D run took 249 s).

**Model:** `pna_finetune_f1.0_s0` from aml-gnn-transfer Phase D: Multi-PNA + EU, pre-trained on HI-Small and fine-tuned on all LI-Small training labels. It was chosen by **validation** F1 (0.332, the best of the fine-tuned runs); its test F1 0.367 is recorded but was not used to choose.

**aml-gnn-transfer is only read.** The notebook clones it fresh into `/content` at the pinned commit and changes nothing there or on its Drive folder, except that a missing graph cache would be rebuilt there. It writes one new file, to `aml_gnn_scores/` on Drive.

**On Drive already** (from the GNN project, under `MyDrive/aml-gnn-transfer/`):
- `phaseD/models/pna_finetune_f1.0_s0/checkpoint_True_finetuned.tar` and `degree_table.json`
- `data/multignn_runs/pna_finetune_f1.0_s0/data_config.json` and `model_settings.json`
- `data/Small_LI/formatted_transactions.csv`
- `graph_cache/Small_LI_*.pt` (optional: rebuilt in several CPU minutes if missing)

**Upload once:** `src/gnn_export.py` from aml-pipeline to `MyDrive/aml_gnn_scores/`.

**Output:** `MyDrive/aml_gnn_scores/gnn_scores.parquet` with columns transaction_id (IBM EdgeID, the pipeline's id), score and period. It has no labels. Download it to `aml-pipeline/artifacts/`.

In [ ]:
# ---- Settings (change here, nowhere else) ----
import os, subprocess, sys, time

GNN_REPO_URL = "https://github.com/coachuttam/aml-gnn-transfer.git"
GNN_COMMIT = "5fe1302"  # Phase D results commit; scoring code unchanged since the run
RUN_ID = "pna_finetune_f1.0_s0"
WORK = "/content"
REPO = f"{WORK}/aml-gnn-transfer"
GNN_DRIVE = "/content/drive/MyDrive/aml-gnn-transfer"   # read only
OUT_DRIVE = "/content/drive/MyDrive/aml_gnn_scores"     # gnn_export.py in, scores out
OUTPUT = f"{OUT_DRIVE}/gnn_scores.parquet"


def run(command):
    """Run a shell command, show its output as it arrives, and stop if it fails."""
    process = subprocess.Popen(["bash", "-o", "pipefail", "-c", command],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed: {command}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

# The same folders the GNN project used, so its code finds the checkpoint, data and graph cache.
os.environ["AML_DATA_ROOT"] = f"{GNN_DRIVE}/data"
os.environ["AML_GRAPH_CACHE"] = f"{GNN_DRIVE}/graph_cache"
os.environ["AML_MODEL_ROOT"] = f"{GNN_DRIVE}/phaseD/models"
os.environ["AML_SOURCE_MODEL_ROOT"] = f"{GNN_DRIVE}/phaseD/models"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

needed = [
    f"{GNN_DRIVE}/phaseD/models/{RUN_ID}/checkpoint_True_finetuned.tar",
    f"{GNN_DRIVE}/phaseD/models/{RUN_ID}/degree_table.json",
    f"{GNN_DRIVE}/data/multignn_runs/{RUN_ID}/data_config.json",
    f"{GNN_DRIVE}/data/multignn_runs/{RUN_ID}/model_settings.json",
    f"{GNN_DRIVE}/data/Small_LI/formatted_transactions.csv",
    f"{OUT_DRIVE}/gnn_export.py",
]
missing = [path for path in needed if not os.path.exists(path)]
assert not missing, "missing on Drive: %s" % missing
assert not os.path.exists(OUTPUT), "already exported: %s (delete it to export again)" % OUTPUT
run(f"ls -la {GNN_DRIVE}/graph_cache | grep Small_LI || echo 'no LI graph cache: it will be built (several CPU minutes)'")

In [ ]:
# A fresh clone at the pinned commit; nothing is pushed or changed in that repo.
run(f"rm -rf {REPO} && git clone -q {GNN_REPO_URL} {REPO}")
run(f"git -C {REPO} checkout -q {GNN_COMMIT} && git -C {REPO} submodule update --init")
run(f"git -C {REPO} log -1 --oneline && git -C {REPO} submodule status")
os.chdir(REPO)

Environment: the same recipe as the GNN project's runner. It uses Multi-GNN's `env.yml` (micromamba, CUDA torch) plus pyg-lib, and falls back to pip wheels. It must pass `src.check_environment`.

In [ ]:
start = time.time()
MICROMAMBA = f"{WORK}/bin/micromamba"
ENV_PYTHON = f"{MICROMAMBA} run -r {WORK}/mamba -n multignn python"
try:
    run(f"curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba")
    run(f"{MICROMAMBA} create -y -r {WORK}/mamba -n multignn -f external/Multi-GNN/env.yml 'pytorch=*=*cuda11.8*'")
    run(f"{ENV_PYTHON} -m src.pyg_wheels --only pyg_lib")
    run(f"{ENV_PYTHON} -m src.check_environment")
    PYTHON = ENV_PYTHON
except RuntimeError as problem:
    print("env.yml not usable:", problem, "- falling back to pip wheels")
    run(f"{sys.executable} -m src.pyg_wheels")
    run(f"{sys.executable} -m src.check_environment")
    PYTHON = sys.executable
run(f"{PYTHON} -m pip install -q pyarrow")  # to write parquet
print(f"Environment ready in {(time.time() - start) / 60:.1f} min")

Score validation and test, then write the parquet. The script prints validation and test F1 recomputed from these scores (threshold from validation) beside the recorded 0.332 / 0.367. "differs" means a different model or data was loaded: do not use the file then.

In [ ]:
start = time.time()
run(f"cd {REPO} && PYTHONPATH={REPO} {PYTHON} {OUT_DRIVE}/gnn_export.py --gnn-repo {REPO} --output {OUTPUT}")
print(f"Export took {(time.time() - start) / 60:.1f} min")

In [ ]:
import pandas as pd
scores = pd.read_parquet(OUTPUT)
print(scores.groupby("period")["score"].describe())
print("rows:", len(scores), "| unique ids:", scores["transaction_id"].nunique())
# Expected: 1,316,442 validation and 1,174,673 test rows (the split in aml-pipeline's data.py).